In [ ]:
# 10_rag_evidence_seed — official remediation playbook chunks -> rag.doc_chunks
# RAG 품질, prompt, embedding, Vector Search index는 DS/AI 영역이다.
# 이 notebook은 DS/AI가 검색 source로 사용할 doc_chunks 테이블만 적재한다.


In [ ]:
dbutils.widgets.text("catalog", "access_drift_dev")
dbutils.widgets.text("rag_doc_chunks_path", "data/dev/rag/doc_chunks.jsonl")
dbutils.widgets.text("run_id", "manual")

CATALOG = dbutils.widgets.get("catalog")
RAG_DOC_CHUNKS_RELATIVE_PATH = dbutils.widgets.get("rag_doc_chunks_path").strip("/")
RUN_ID = dbutils.widgets.get("run_id")
RAG_DOC_CHUNKS_LOCAL_PATH = f"/Workspace/Shared/access-drift/files/{RAG_DOC_CHUNKS_RELATIVE_PATH}"
RAG_DOC_CHUNKS_PATH = f"file:{RAG_DOC_CHUNKS_LOCAL_PATH}"


In [ ]:
import os

if not os.path.isfile(RAG_DOC_CHUNKS_LOCAL_PATH):
    raise RuntimeError(
        f"RAG doc_chunks seed 파일을 찾을 수 없습니다: {RAG_DOC_CHUNKS_LOCAL_PATH}\n"
        "bundle deploy가 완료됐는지 확인하세요."
    )
print(f"rag doc_chunks seed path OK: {RAG_DOC_CHUNKS_LOCAL_PATH}")


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

seed_schema = T.StructType([
    T.StructField("chunk_id", T.StringType(), False),
    T.StructField("doc_id", T.StringType(), False),
    T.StructField("source_system", T.StringType(), False),
    T.StructField("source_area", T.StringType(), False),
    T.StructField("doc_type", T.StringType(), False),
    T.StructField("drift_type", T.StringType(), False),
    T.StructField("principal_kind", T.StringType(), False),
    T.StructField("action_type", T.StringType(), False),
    T.StructField("chunk_order", T.IntegerType(), False),
    T.StructField("source_title", T.StringType(), False),
    T.StructField("source_uri", T.StringType(), False),
    T.StructField("effective_date", T.StringType(), False),
    T.StructField("chunk_text", T.StringType(), False),
])

required_columns = [field.name for field in seed_schema]
raw_df = spark.read.schema(seed_schema).json(RAG_DOC_CHUNKS_PATH)
missing_columns = [col for col in required_columns if col not in raw_df.columns]
if missing_columns:
    raise RuntimeError(f"RAG doc_chunks seed missing columns: {missing_columns}")

invalid_principal_kinds = raw_df.filter(~F.col("principal_kind").isin("nhi", "hi")).count()
assert invalid_principal_kinds == 0, f"invalid principal_kind rows: {invalid_principal_kinds}"

df = raw_df.select(*required_columns).withColumn(
    "effective_date", F.to_date("effective_date")
).withColumn(
    "chunk_hash", F.sha2(F.col("chunk_text"), 256)
).withColumn(
    "generated_at", F.current_timestamp()
).withColumn(
    "run_id", F.lit(RUN_ID)
)

df.select(
    "chunk_id", "doc_id", "source_system", "source_area", "doc_type", "drift_type",
    "principal_kind", "action_type", "chunk_text", "chunk_hash", "chunk_order",
    "source_title", "source_uri", "effective_date", "generated_at", "run_id",
).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.rag.doc_chunks")

chunk_count = spark.table(f"{CATALOG}.rag.doc_chunks").count()
assert chunk_count > 0, "rag.doc_chunks has no rows"
print(f"rag.doc_chunks: {chunk_count} rows")


In [ ]:
duplicate_chunks = spark.sql(f"""
  SELECT COUNT(*) AS cnt
  FROM (
    SELECT chunk_id
    FROM {CATALOG}.rag.doc_chunks
    GROUP BY chunk_id
    HAVING COUNT(*) > 1
  )
""").first()["cnt"]
assert duplicate_chunks == 0, f"duplicate chunk_id: {duplicate_chunks}"

forbidden_pattern = "(?i)(secret value|token value|private key value|BEGIN PRIVATE KEY)"
forbidden_chunks = spark.sql(f"""
  SELECT COUNT(*) AS cnt
  FROM {CATALOG}.rag.doc_chunks
  WHERE chunk_text RLIKE '{forbidden_pattern}'
""").first()["cnt"]
assert forbidden_chunks == 0, f"forbidden value pattern in doc_chunks: {forbidden_chunks}"

spark.table(f"{CATALOG}.rag.doc_chunks").select(
    "source_system", "source_area", "doc_type", "drift_type", "principal_kind",
    "action_type", "source_title", "source_uri"
).orderBy("source_system", "source_area", "chunk_id").show(truncate=False)

summary = [
    "rag doc_chunks seed smoke test passed",
    f"doc_chunks: {chunk_count} rows",
    f"duplicate_chunk_id: {duplicate_chunks}",
    f"forbidden_chunks: {forbidden_chunks}",
]
dbutils.notebook.exit("\n".join(summary))
